[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nammaschool/interactive-learning/blob/main/notebooks/courses/wave_optics.ipynb)

# Wave optics full course: the practice set, in Python

Twenty-eight exercises in three tiers: **tier 1** warm-ups (one idea, one step),
**tier 2** core problems (the level of the course's worked problems) and **tier 3** stretch problems (run a formula
backwards, combine two ideas, spot the trap). Each tier is harder than the last.

This notebook also has the course's **wave pictures**, computed the way the course computed them: Huygens' wavelets
at a glass surface, and patterns made by adding one small arrow for every wavelet and squaring the sum. You can move
all of it yourself: the **angle** of a wave meeting glass, the **phase** between two waves, the **spacing** of two
dippers in a ripple tank, the **wavelength**, **slit spacing** and **screen distance** of two slits, the **width** of
one slit, the **angles** of three polaroids, and the laser measurement of the course's opening question.

**How to use this notebook.** The grey boxes are cells. Click the first one and press **Shift + Enter** (or the ▶ Run
button) to run it. Run each cell in turn, from the top down. When sliders appear, move them and watch: the picture
redraws by itself. Change any number and run the cell again: nothing can break. It works in your browser (no install)
and in Google Colab.

**Typing answers.** Work the question out first (on paper, or in a cell of your own), then type your answer into the
box and press **check**: it tells you which parts are right. Press **show working** to see the steps. Give the answer
in the unit the question asks for, to two decimals or more. You can type the arithmetic: `600e-9*2/0.5e-3*1000`,
`asind(sind(50)/1.33)`, `4*cosd(45)**2`, `10*cosd(60)**2`. The functions `sind`, `cosd`, `tand` take degrees and
`asind`, `acosd` give degrees; `sin`, `cos`, `tan`, `asin` and `sqrt` work in radians, with `pi`. For several answers,
separate them with commas, in the order the question lists them (the box's label shows the order). You can also put
your answer between the quotes in a cell's last line, `practice("P1.5", "")`, and run the cell: it checks straight
away.

**Units.** Put every length in metres before you use a formula: 1 mm = 10⁻³ m (`1e-3`), 1 nm = 10⁻⁹ m (`1e-9`). Then
turn the answer into the unit the question asks for.

**Laser safety.** If you try the two-slit measurement for real: never look into a laser beam, never aim it at
anyone, and use a low-power pointer.

**The cheat sheet.** Every formula and value from the course on two pages:
[wave optics cheat sheet (PDF)](https://nammaschool.github.io/interactive-learning/cheatsheets/wave_optics_cheat_sheet.pdf).

In [ ]:
# Setup: in the browser version (JupyterLite) the sliders need one install; elsewhere this does nothing.
import sys
if "pyodide" in sys.modules:
    import piplite
    await piplite.install(["ipywidgets"])

In [ ]:
# The look, the constants, the formulas, the pattern sums and the checker: run this cell first.
import ast, math, operator, re, textwrap
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
from ipywidgets import interact, FloatSlider, IntSlider, Text, Button, Output, HBox, VBox, Layout

SLATE = "#1f2b28"                                     # the slate ground
CHALK, DIM, GHOST = "#ece6d6", "#93a097", "#3b4c47"   # wavefronts and lines in chalk, labels dim, ghost lines
SKY, MINT, CORAL, VIOLET, GOLD = "#6fb6bb", "#a0c27c", "#e3896b", "#b49ad6", "#e8c56b"
# wavefronts CHALK · the wave from slit 1 CORAL, from slit 2 MINT · intensity VIOLET · the one mark GOLD

plt.rcParams.update({
    "figure.facecolor": SLATE, "axes.facecolor": SLATE, "savefig.facecolor": SLATE,
    "text.color": CHALK, "axes.labelcolor": CHALK, "axes.edgecolor": DIM, "axes.titlecolor": CHALK,
    "xtick.color": DIM, "ytick.color": DIM, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.axisbelow": True, "grid.color": GHOST, "grid.linewidth": 0.8,
    "font.size": 13, "axes.titlesize": 15, "axes.labelsize": 13, "legend.fontsize": 12,
    "lines.linewidth": 2.4, "figure.dpi": 100,
})

def footnote(fig, text="summed wavelet by wavelet · checked against the formula"):
    fig.text(0.99, 0.01, text, color=DIM, fontsize=10, ha="right", va="bottom", family="monospace")

# ---- the constants (the course's rounded values; the same in every question) ----
C = 3.00e8                         # m/s, the speed of light
N_AIR, N_WATER, N_GLASS = 1.0, 1.33, 1.5              # refractive indices
NM, UM, MM, CM = 1e-9, 1e-6, 1e-3, 1e-2               # a nanometre, a micrometre, a millimetre, a centimetre, in metres
RED, GREEN = 650 * NM, 532 * NM                       # a red and a green laser pointer

def sind(d): return math.sin(math.radians(d))
def cosd(d): return math.cos(math.radians(d))
def tand(d): return math.tan(math.radians(d))
def asind(x): return math.degrees(math.asin(x))
def acosd(x): return math.degrees(math.acos(x))
def close(a, b, rel=1e-4):
    """True when a computed value matches the formula (to a ten-thousandth by default)."""
    return abs(a - b) <= rel * max(abs(a), abs(b))

# ---------- the course's formulas (lengths in metres, angles in degrees) ----------
def frequency(lam, v=C): return v / lam                                          # v = f λ
def speed_in(n): return C / n                                                    # n = c/v
def lam_in(lam, n): return lam / n                                               # the frequency stays; the wavelength shrinks
def refract(theta1, n1, n2):                                                     # n₁ sin i = n₂ sin r (None: no refracted wave)
    s = n1 * sind(theta1) / n2
    return None if s > 1 else asind(s)
def critical(n1, n2=1.0): return asind(n2 / n1)
def phase_deg(path, lam): return 360.0 * path / lam                              # a path difference of λ is a whole cycle
def two_wave(phi_deg): return 4 * cosd(phi_deg / 2) ** 2                         # I/I₀ = 4 cos²(φ/2), two equal waves
def path_diff(x, d, D): return x * d / D                                         # S₂P − S₁P ≈ x d / D
def fringe_width(lam, D, d): return lam * D / d                                  # β = λD/d
def bright(n, lam, D, d): return n * lam * D / d                                 # x = nλD/d
def dark(n, lam, D, d): return (n + 0.5) * lam * D / d                           # x = (n + ½)λD/d
def slit_dark(n, lam, D, a): return n * lam * D / a                              # a single slit's dark lines: x = nλD/a
def central_width(lam, D, a): return 2 * lam * D / a                             # its central maximum
def slit_shape(x, lam, D, a):                                                    # (sin α/α)², α = π a x/(λ D)
    al = np.pi * a * np.asarray(x, float) / (lam * D)
    return np.sinc(al / np.pi) ** 2
def malus(theta_deg): return cosd(theta_deg) ** 2                                # I/I₀ = cos² θ

# ---- Huygens' construction at a flat surface (the course's, computed) ----
def huygens(i, n1, n2, BC=3.0):
    """A plane wavefront AB meets a flat surface (y = 0) at angle of incidence i, in material n1 above and n2 below.
    While B travels to C (a distance BC in material 1), the wavelet from A grows to AE = BC × n1/n2 in material 2. The
    new wavefront is the tangent from C to that wavelet; r is its angle to the surface (None: the wavelet has passed C,
    no tangent: total internal reflection)."""
    AC = BC / sind(i)
    AE = BC * n1 / n2
    out = {"A": (0.0, 0.0), "B": (AC - BC * sind(i), BC * cosd(i)), "C": (AC, 0.0), "AC": AC, "BC": BC, "AE": AE}
    if AE < AC:
        out["E"] = (AE * AE / AC, -AE * math.sqrt(1 - (AE / AC) ** 2)); out["r"] = asind(AE / AC)
    else:
        out["E"] = None; out["r"] = None
    return out

# ---- patterns, summed: every slit a row of Huygens sources; exact distances; arrows added, then squared ----
def slit_sources(centres, a=0.0, n=1):
    """Point sources standing for slits centred at `centres` (heights on the slit card), each of width a cut into n
    equal strips (a = 0 or n = 1: one source per slit)."""
    if a <= 0 or n == 1: return list(centres)
    return [c_ - a / 2 + a * (k + 0.5) / n for c_ in centres for k in range(n)]

def pattern(xs, sources, lam, D):
    """The intensity at each height x on a screen a distance D away, as a fraction of the intensity at x = 0: one
    arrow (a phase) per source, from its exact distance; the arrows added; the sum squared."""
    xs = np.asarray(xs, float); src = np.asarray(sources, float)
    cyc = np.hypot(D, xs[:, None] - src[None, :]) / lam
    amp = np.exp(2j * np.pi * (cyc - np.floor(cyc))).sum(axis=1)
    cyc0 = np.hypot(D, src) / lam
    amp0 = np.exp(2j * np.pi * (cyc0 - np.floor(cyc0))).sum()
    return np.abs(amp) ** 2 / abs(amp0) ** 2

def peaks(xs, I, floor=0.0):
    """Local maxima of a sampled pattern, each refined by a parabola through three samples: [(x, height)]."""
    out = []
    for k in range(1, len(xs) - 1):
        if I[k] > I[k - 1] and I[k] >= I[k + 1] and I[k] > floor:
            den = I[k - 1] - 2 * I[k] + I[k + 1]
            sh = 0.0 if den == 0 else 0.5 * (I[k - 1] - I[k + 1]) / den
            out.append((xs[k] + sh * (xs[k + 1] - xs[k]), I[k] - 0.25 * (I[k - 1] - I[k + 1]) * sh))
    return out

def dips(xs, I):
    """Local minima of a sampled pattern: [(x, height)]."""
    return [(x_, -h_) for x_, h_ in peaks(xs, [-v_ for v_ in I], floor=-1e9)]

def strip(ax, xs_mm, I, colour=CHALK, gamma=0.5):
    """A pattern as you would see it on a screen: a band whose brightness is the intensity (gamma < 1 lifts the faint
    parts, as your eye does)."""
    from matplotlib.colors import LinearSegmentedColormap
    cmap = LinearSegmentedColormap.from_list("light", [SLATE, colour])
    ax.imshow(np.clip(np.asarray(I), 0, 1)[None, :] ** gamma, aspect="auto", cmap=cmap, vmin=0, vmax=1,
              extent=[xs_mm[0], xs_mm[-1], 0, 1])
    ax.set_yticks([]); ax.grid(False)

# ---- reading a typed answer ----
OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv,
       ast.Pow: operator.pow, ast.USub: operator.neg, ast.UAdd: operator.pos}
FUNCS = {"sqrt": math.sqrt, "cos": math.cos, "sin": math.sin, "tan": math.tan, "asin": math.asin, "acos": math.acos,
         "atan": math.atan, "sind": sind, "cosd": cosd, "tand": tand, "asind": asind, "acosd": acosd}

def calc(text):
    """Plain arithmetic only: numbers, + − × ÷, powers, brackets, pi, sqrt, sin, cos, tan, asin (radians) and sind,
    cosd, tand, asind, acosd (degrees)."""
    def ev(n):
        if isinstance(n, ast.Expression): return ev(n.body)
        if isinstance(n, ast.Constant) and isinstance(n.value, (int, float)): return n.value
        if isinstance(n, ast.Name) and n.id in ("pi", "π"): return math.pi
        if isinstance(n, ast.BinOp) and type(n.op) in OPS: return OPS[type(n.op)](ev(n.left), ev(n.right))
        if isinstance(n, ast.UnaryOp) and type(n.op) in OPS: return OPS[type(n.op)](ev(n.operand))
        if isinstance(n, ast.Call) and getattr(n.func, "id", "") in FUNCS and len(n.args) == 1:
            return FUNCS[n.func.id](ev(n.args[0]))
        raise ValueError(text)
    return float(ev(ast.parse(text, mode="eval")))

UNIT = r"\s*(W/m²|W/m2|m/s|Hz|nm|cm|mm|m|I₀|I0|°|degrees?|wavelengths?|fringes?|times|×)\s*$"

def read_part(s):
    s = s.strip().replace("−", "-").replace("–", "-").replace("×", "*").replace("÷", "/").replace("^", "**")
    s = s.replace("√", "sqrt")
    t = re.sub(UNIT, "", s).strip()
    try: return calc(t)
    except Exception: return s

def right(got, want, tol):
    return not isinstance(got, str) and abs(got - want) <= max(tol, 1e-7)

def show(v):
    return f"{v:,.10g}".replace(",", " ").replace("-", "−")

# ---- the exercises: the practice set's own (the course's questions and answers) ----
# @EX
EX = [
    {"id": 'P1.1', "tier": 1, "group": 'Wavefronts and Huygens',
     "q": 'Green light has a wavelength of 500 nm in air. Taking c = 3.00 × 10⁸ m/s, what is its frequency, in units of 10¹⁴ Hz?',
     "parts": ['frequency (× 10¹⁴ Hz)'],
     "answer": [5.999999999999999], "tol": [0.005], "function": '—',
     "hint": 'v = f λ, so f = c ÷ λ, with λ in metres',
     "working": 'f = c/λ = 3.00 × 10⁸ / (500 × 10⁻⁹) = 6.00 × 10¹⁴ Hz'},
    {"id": 'P1.2', "tier": 1, "group": 'Wavefronts and Huygens',
     "q": 'Light of wavelength 600 nm in air enters glass of refractive index 1.5. What is its wavelength inside the glass, in nm?',
     "parts": ['wavelength (nm)'],
     "answer": [400.0], "tol": [0.005], "function": '—',
     "hint": "the frequency can't change, so the wavelength shrinks by n",
     "working": 'the frequency stays the same and the speed falls to c/n, so λ = 600 / 1.5 = 400 nm'},
    {"id": 'P1.3', "tier": 1, "group": 'Wavefronts and Huygens',
     "q": 'A plane wavefront in air meets a flat water surface (n = 1.33) with an angle of incidence of 50°. What angle does the refracted wavefront make with the surface, in degrees?',
     "parts": ['angle (°)'],
     "answer": [35.16781910065805], "tol": [0.005], "function": 'sin, asin',
     "hint": "Huygens' construction gives Snell's law: n₁ sin i = n₂ sin r",
     "working": 'sin r = sin 50° / 1.33 = 0.5760, so r = 35.17°'},
    {"id": 'P1.4', "tier": 1, "group": 'Interference',
     "q": 'Two waves of wavelength 500 nm reach a point with a path difference of 1250 nm. How many wavelengths is that?',
     "parts": ['path difference (wavelengths)'],
     "answer": [2.5], "tol": [0.005], "function": '—',
     "hint": 'divide the path difference by the wavelength',
     "working": '1250 / 500 = 2.5 wavelengths: a whole number plus a half, so the waves cancel (dark)'},
    {"id": 'P1.5', "tier": 1, "group": 'Interference',
     "q": 'In a two-slit experiment light of 600 nm passes slits 0.5 mm apart and falls on a screen 2 m away. What is the fringe width, in mm?',
     "parts": ['fringe width (mm)'],
     "answer": [2.4000000000000004], "tol": [0.005], "function": '—',
     "hint": 'β = λD/d, everything in metres first',
     "working": 'β = λD/d = 600 × 10⁻⁹ × 2 / (0.5 × 10⁻³) = 2.4 × 10⁻³ m = 2.4 mm'},
    {"id": 'P1.6', "tier": 1, "group": 'Interference',
     "q": 'Two equal coherent waves, each of intensity I₀, meet with a phase difference of 90°. What is the intensity there, in units of I₀?',
     "parts": ['intensity (I₀)'],
     "answer": [2.0000000000000004], "tol": [0.005], "function": 'cos',
     "hint": 'I = 4 I₀ cos²(φ/2)',
     "working": 'I = 4 I₀ cos²(φ/2) = 4 × cos² 45° = 4 × 0.5 = 2 I₀'},
    {"id": 'P1.7', "tier": 1, "group": 'Diffraction',
     "q": 'Light of 500 nm passes a single slit 0.1 mm wide. A screen is 2 m away. How far from the centre is the first dark line, in mm?',
     "parts": ['distance (mm)'],
     "answer": [10.000000000000002], "tol": [0.005], "function": '—',
     "hint": 'the first dark line is at the angle λ/a, so x = λD/a',
     "working": 'x = λD/a = 500 × 10⁻⁹ × 2 / (0.1 × 10⁻³) = 10 × 10⁻³ m = 10 mm'},
    {"id": 'P1.8', "tier": 1, "group": 'Diffraction',
     "q": 'For that slit (500 nm, 0.1 mm wide, a screen 2 m away), how wide is the central maximum, in mm?',
     "parts": ['width (mm)'],
     "answer": [20.000000000000004], "tol": [0.005], "function": '—',
     "hint": 'twice the distance to the first dark line',
     "working": 'from the first dark line on one side to the first on the other: 2λD/a = 2 × 10 mm = 20 mm'},
    {"id": 'P1.9', "tier": 1, "group": 'Polarisation',
     "q": "Polarised light of intensity 10 W/m² meets a polaroid whose pass axis is at 60° to the light's vibration. What intensity gets through, in W/m²?",
     "parts": ['intensity (W/m²)'],
     "answer": [2.500000000000001], "tol": [0.005], "function": 'cos',
     "hint": "Malus's law: I = I₀ cos² θ",
     "working": 'I = I₀ cos² θ = 10 × cos² 60° = 10 × 0.25 = 2.5 W/m²'},
    {"id": 'P1.10', "tier": 1, "group": 'Laser and slits',
     "q": 'A red laser pointer (650 nm) shines through two slits 0.5 mm apart onto a wall 3 m away. What is the fringe width, in mm?',
     "parts": ['fringe width (mm)'],
     "answer": [3.9], "tol": [0.005], "function": '—',
     "hint": 'β = λD/d',
     "working": 'β = λD/d = 650 × 10⁻⁹ × 3 / (0.5 × 10⁻³) = 3.9 × 10⁻³ m = 3.9 mm'},
    {"id": 'P2.1', "tier": 2, "group": 'Wavefronts and Huygens',
     "q": 'A plane wavefront travelling in water (n = 1.33) meets the surface with an angle of incidence of 35°. What angle does the wavefront in the air make with the surface (degrees), and what is the critical angle for water (degrees)?',
     "parts": ['angle in air (°)', 'critical angle (°)'],
     "answer": [49.716686108898614, 48.753466631327235], "tol": [0.005, 0.005], "function": 'sin, asin',
     "hint": 'n₁ sin i = n₂ sin r with n₁ = 1.33; at the critical angle r = 90°',
     "working": 'sin r = 1.33 × sin 35° = 0.7629, so r = 49.72°: the wavelets are bigger in air, so the front swings away; sin θc = 1/1.33, θc = 48.75°'},
    {"id": 'P2.2', "tier": 2, "group": 'Wavefronts and Huygens',
     "q": 'Yellow light of wavelength 589 nm in air enters water (n = 1.33). What is its speed in the water (in units of 10⁸ m/s), and its wavelength there (nm)?',
     "parts": ['speed (× 10⁸ m/s)', 'wavelength (nm)'],
     "answer": [2.255639097744361, 442.85714285714283], "tol": [0.005, 0.005], "function": '—',
     "hint": 'both the speed and the wavelength are divided by n',
     "working": 'v = c/n = 3.00 × 10⁸ / 1.33 = 2.26 × 10⁸ m/s; λ = 589 / 1.33 = 442.86 nm; the frequency is unchanged'},
    {"id": 'P2.3', "tier": 2, "group": 'Interference',
     "q": 'Two coherent sources send out waves of wavelength 2 cm. A point is 23 cm from one source and 30 cm from the other. What is the phase difference there, in degrees, after whole cycles are removed (0 to 360)?',
     "parts": ['phase difference (°)'],
     "answer": [180.0], "tol": [0.005], "function": '—',
     "hint": 'one wavelength of path is 360° of phase; drop the whole wavelengths',
     "working": 'path difference 30 − 23 = 7 cm = 3.5 wavelengths; the half wavelength left over is half a cycle: 180°, so the waves cancel'},
    {"id": 'P2.4', "tier": 2, "group": 'Interference',
     "q": 'Light of 500 nm passes two slits 0.25 mm apart; the screen is 1.5 m away. How far from the centre is the fourth bright fringe (mm), and the second dark fringe (mm)? (Count the dark fringes from the centre: first, second, …)',
     "parts": ['fourth bright (mm)', 'second dark (mm)'],
     "answer": [12.000000000000002, 4.5], "tol": [0.005, 0.005], "function": '—',
     "hint": 'bright at nβ; dark halfway between: (n + ½)β',
     "working": 'β = λD/d = 3 mm; the fourth bright fringe is at 4β = 12 mm; the dark fringes sit at ½β, 1½β, …, so the second is at 1.5 × 3 = 4.5 mm'},
    {"id": 'P2.5', "tier": 2, "group": 'Interference',
     "q": 'Two equal coherent waves, each of intensity I₀, meet where their path difference is a third of a wavelength. What is the intensity there, in units of I₀?',
     "parts": ['intensity (I₀)'],
     "answer": [1.0000000000000004], "tol": [0.005], "function": 'cos',
     "hint": 'turn the path difference into a phase difference first',
     "working": 'a third of a wavelength is 360°/3 = 120° of phase; I = 4 I₀ cos² 60° = 4 × 0.25 = 1 I₀'},
    {"id": 'P2.6', "tier": 2, "group": 'Interference',
     "q": 'A two-slit experiment gives fringes 2.6 mm wide in air. The whole apparatus is lowered into water (n = 1.33). What is the fringe width now, in mm?',
     "parts": ['fringe width (mm)'],
     "answer": [1.9548872180451127], "tol": [0.005], "function": '—',
     "hint": 'the frequency stays the same, so the wavelength, and with it β, is divided by n',
     "working": 'in water the wavelength is λ/n, so β becomes β/n = 2.6 / 1.33 = 1.95 mm'},
    {"id": 'P2.7', "tier": 2, "group": 'Diffraction',
     "q": 'Light of 650 nm passes a single slit 0.13 mm wide onto a screen 2 m away. How wide is the central bright band (mm), and how far from the centre is the second dark line (mm)?',
     "parts": ['central band (mm)', 'second dark line (mm)'],
     "answer": [19.999999999999996, 19.999999999999996], "tol": [0.005, 0.005], "function": '—',
     "hint": 'dark lines at nλD/a; the central band spans −1 to +1',
     "working": 'λD/a = 650 × 10⁻⁹ × 2 / (0.13 × 10⁻³) = 10 mm; the central band is 2 × 10 = 20 mm wide; the second dark line is at 2λD/a = 20 mm'},
    {"id": 'P2.8', "tier": 2, "group": 'Diffraction',
     "q": 'Two slits are 0.4 mm apart and each is 0.1 mm wide. How many bright two-slit fringes lie inside the central band of the single-slit pattern?',
     "parts": ['number of fringes'],
     "answer": [7.0], "tol": [0.005], "function": '—',
     "hint": 'fringe n sits at nλD/d; the band ends at λD/a; compare d with a',
     "working": "d/a = 4, so the fourth fringe on each side falls on the single slit's first dark line and is missing; inside are n = −3 … 3: seven fringes"},
    {"id": 'P2.9', "tier": 2, "group": 'Polarisation',
     "q": 'Unpolarised light of intensity 20 W/m² passes through two polaroids whose pass axes are 30° apart. What intensity leaves the second, in W/m²?',
     "parts": ['intensity (W/m²)'],
     "answer": [7.500000000000001], "tol": [0.005], "function": 'cos',
     "hint": "half through the first; then Malus's law",
     "working": 'the first polaroid passes half: 10 W/m²; the second passes cos² 30° = 0.75 of that: 7.5 W/m²'},
    {"id": 'P2.10', "tier": 2, "group": 'Laser and slits',
     "q": "A laser shines through two slits 0.25 mm apart onto a wall 2 m away. Ten fringe gaps measure 43 mm. What is the laser's wavelength, in nm?",
     "parts": ['wavelength (nm)'],
     "answer": [537.5], "tol": [0.005], "function": '—',
     "hint": 'measure many gaps and divide; then λ = βd/D',
     "working": 'β = 43 / 10 = 4.3 mm; λ = βd/D = 4.3 × 10⁻³ × 0.25 × 10⁻³ / 2 = 5.375 × 10⁻⁷ m = 537.5 nm: a green pointer'},
    {"id": 'P3.1', "tier": 3, "group": 'Wavefronts and Huygens',
     "q": "Light has a wavelength of 640 nm in air and 400 nm inside a clear liquid. What is the liquid's refractive index, and the speed of light in it (in units of 10⁸ m/s)?",
     "parts": ['refractive index', 'speed (× 10⁸ m/s)'],
     "answer": [1.6, 1.875], "tol": [0.005, 0.005], "function": '—',
     "hint": 'same frequency in both: the wavelengths are in the ratio of the speeds',
     "working": 'the frequency is the same in both, so n = λ(air)/λ(liquid) = 640/400 = 1.6; v = c/n = 3.00 × 10⁸ / 1.6 = 1.875 × 10⁸ m/s'},
    {"id": 'P3.2', "tier": 3, "group": 'Wavefronts and Huygens',
     "q": 'Inside glass (n = 1.5) a refracted wavefront makes 25° with the surface. What was the angle of incidence in the air, in degrees?',
     "parts": ['angle of incidence (°)'],
     "answer": [39.3404765044617], "tol": [0.005], "function": 'sin, asin',
     "hint": "Snell's law backwards",
     "working": 'sin i = n sin r = 1.5 × sin 25° = 0.6339, so i = 39.34°'},
    {"id": 'P3.3', "tier": 3, "group": 'Interference',
     "q": 'In a two-slit experiment with slits 0.3 mm apart and a screen 1.2 m away, the fifth bright fringe is 11 mm from the centre. What is the wavelength, in nm?',
     "parts": ['wavelength (nm)'],
     "answer": [549.9999999999999], "tol": [0.005], "function": '—',
     "hint": 'the fifth bright fringe is five fringe widths out',
     "working": '5β = 11 mm, so β = 2.2 mm; λ = βd/D = 2.2 × 10⁻³ × 0.3 × 10⁻³ / 1.2 = 5.5 × 10⁻⁷ m = 550 nm'},
    {"id": 'P3.4', "tier": 3, "group": 'Interference',
     "q": 'Two coherent waves have amplitudes in the ratio 3 to 1. What is the ratio of the greatest intensity to the least in their interference pattern?',
     "parts": ['ratio'],
     "answer": [4.0], "tol": [0.005], "function": '—',
     "hint": 'amplitudes add or subtract; intensity is amplitude squared',
     "working": 'at a bright fringe the amplitudes add: 3 + 1 = 4; at a dark fringe they subtract: 3 − 1 = 2; intensity goes as amplitude squared: 16 to 4, a ratio of 4'},
    {"id": 'P3.5', "tier": 3, "group": 'Interference',
     "q": 'A two-slit pattern is made with red light of 650 nm, and then with blue light of 450 nm, on the same apparatus. The red fringes are 3.9 mm wide. How wide are the blue ones, in mm?',
     "parts": ['fringe width (mm)'],
     "answer": [2.7], "tol": [0.005], "function": '—',
     "hint": 'same D and d: β goes as λ',
     "working": 'β is proportional to λ: 3.9 × 450/650 = 2.7 mm'},
    {"id": 'P3.6', "tier": 3, "group": 'Diffraction',
     "q": "A single slit lit with 600 nm light gives a central maximum 18 mm wide on a screen 3 m away. What is the slit's width, in mm?",
     "parts": ['slit width (mm)'],
     "answer": [0.19999999999999998], "tol": [0.005], "function": '—',
     "hint": "run the central band's width backwards",
     "working": 'width = 2λD/a, so a = 2λD/width = 2 × 600 × 10⁻⁹ × 3 / (18 × 10⁻³) = 2 × 10⁻⁴ m = 0.2 mm'},
    {"id": 'P3.7', "tier": 3, "group": 'Polarisation',
     "q": "Unpolarised light passes through three polaroids. The second's pass axis is at 30° to the first's, and the third's is at 90° to the first's. What fraction of the original intensity leaves the third?",
     "parts": ['fraction'],
     "answer": [0.09375000000000006], "tol": [0.005], "function": 'cos',
     "hint": "each polaroid's angle is measured from the light reaching it",
     "working": 'first: 1/2; second: × cos² 30° = 0.75; third (60° from the second): × cos² 60° = 0.25; in all 0.5 × 0.75 × 0.25 = 0.09375 ≈ 0.09'},
    {"id": 'P3.8', "tier": 3, "group": 'Laser and slits',
     "q": 'A red laser pointer (650 nm) lights two slits 0.25 mm apart, each 0.05 mm wide; the wall is 2.5 m away. What is the fringe width (mm), and how many bright fringes lie inside the central band of the single-slit pattern?',
     "parts": ['fringe width (mm)', 'number of fringes'],
     "answer": [6.5, 9.0], "tol": [0.005, 0.005], "function": '—',
     "hint": 'β = λD/d; then compare the slit spacing with the slit width',
     "working": 'β = λD/d = 650 × 10⁻⁹ × 2.5 / (0.25 × 10⁻³) = 6.5 mm; d/a = 5, so the fifth fringe on each side is missing: n = −4 … 4, nine fringes'},
]
# @END EX
Q = {e["id"]: e for e in EX}
TIER_NAMES = {1: "Warm-up", 2: "Core", 3: "Stretch"}
assert len(EX) == 28 and [sum(e["tier"] == t for e in EX) for t in TIER_NAMES] == [10, 10, 8]

def practice(k, answer=""):
    """Exercise k ("P1.4"): an answer box, a check button and a show-working button."""
    q = Q[k]
    n = len(q["answer"])
    box = Text(value=answer, placeholder="your answer" if n == 1 else f"{n} answers, separated by commas",
               layout=Layout(width="520px"))
    check, work, out = Button(description="check"), Button(description="show working"), Output()
    def on_check(_=None):
        out.clear_output()
        with out:
            got = [read_part(p) for p in re.split(r"[,;]", box.value) if p.strip()]
            if len(got) != n:
                print(f"✗ this one asks for {n} answer(s), in this order: {' · '.join(q['parts'])}. You typed {len(got)}."); return
            ok = [right(g_, w, t) for g_, w, t in zip(got, q["answer"], q["tol"])]
            if all(ok): print("✓ right" if n == 1 else "✓ all right"); return
            for good, part in zip(ok, q["parts"]):
                print(f"  {'✓' if good else '✗'} {part}" + ("" if good else ": not quite"))
            print(textwrap.fill(f"Hint: {q['hint']}", 110))
    def on_work(_=None):
        out.clear_output()
        with out:
            print(textwrap.fill(q["working"], 110))
            print("Answer: " + " · ".join(f"{p}: {show(a)}" for p, a in zip(q["parts"], q["answer"])))
    check.on_click(on_check); work.on_click(on_work)
    print(textwrap.fill(f"{k}. {q['q']}", 110))
    if n > 1: print(textwrap.fill("Answers, in order: " + " · ".join(q["parts"]), 110))
    display(VBox([HBox([box, check, work]), out]))
    if answer: on_check()

def answers(k):
    """The answers the practice set gives for exercise k (for the Python cells below to check against)."""
    return Q[k]["answer"]

def agrees(k, got):
    """True when every value in got is within exercise k's tolerance of its answer."""
    return len(got) == len(answers(k)) and all(right(g_, w, t) for g_, w, t in zip(got, answers(k), Q[k]["tol"]))

assert round(fringe_width(600 * NM, 1.5, 0.3 * MM) / MM, 9) == 3.0 and round(refract(45.0, 1.0, 1.5), 2) == 28.13
print("Ready: c =", C, "m/s · n(water) =", N_WATER, "· n(glass) =", N_GLASS, "· a red laser pointer:", round(RED / NM), "nm")

## Tier 1 · Warm-up

One idea, one step: a frequency from a wavelength, the wavelength inside glass, a refracted wavefront, a path
difference in wavelengths, a fringe width, two waves added, a single slit's first dark line and central maximum,
Malus's law, and a laser's fringes.

In [ ]:
practice("P1.1")

In [ ]:
practice("P1.2")

In [ ]:
practice("P1.3")

In [ ]:
practice("P1.4")

In [ ]:
practice("P1.5")

In [ ]:
practice("P1.6")

In [ ]:
practice("P1.7")

In [ ]:
practice("P1.8")

In [ ]:
practice("P1.9")

In [ ]:
practice("P1.10")

### How you'd do this in Python

Each formula from the course is one short function in the first cell: `frequency(lam)`, `lam_in(lam, n)`,
`refract(i, n1, n2)`, `two_wave(phi)`, `fringe_width(lam, D, d)`, `slit_dark(n, lam, D, a)`,
`central_width(lam, D, a)`, `malus(theta)` and the rest. Lengths go in as metres (`600 * NM`, `0.5 * MM`); divide by
`MM` or `NM` to read the answer in millimetres or nanometres. Each line below computes one warm-up answer and checks
it against the practice set.

In [ ]:
print(f"P1.1  f = {frequency(500 * NM):.3g} Hz;  P1.5  β = {fringe_width(600 * NM, 2.0, 0.5 * MM) / MM:.2f} mm;  "
      f"P1.9  I = {10 * malus(60):.2f} W/m²")
assert agrees("P1.1", [frequency(500 * NM) / 1e14]) and agrees("P1.2", [lam_in(600 * NM, 1.5) / NM])
assert agrees("P1.3", [refract(50, 1, N_WATER)]) and agrees("P1.4", [1250 / 500]) and agrees("P1.5", [fringe_width(600 * NM, 2.0, 0.5 * MM) / MM])
assert agrees("P1.6", [two_wave(90)]) and agrees("P1.7", [slit_dark(1, 500 * NM, 2.0, 0.1 * MM) / MM])
assert agrees("P1.8", [central_width(500 * NM, 2.0, 0.1 * MM) / MM]) and agrees("P1.9", [10 * malus(60)])
assert agrees("P1.10", [fringe_width(RED, 3.0, 0.5 * MM) / MM])
print("Every tier 1 answer, worked out in Python, matches the practice set.")

## Huygens' wavelets at a surface

A plane wavefront AB meets a flat surface at a slant. End A touches first. While end B travels on to the surface at
C, the point A sends a wavelet into the second material, at that material's speed. The new wavefront is the line from
C that just touches the wavelet. Nothing else is assumed, and the angle that comes out is Snell's law: the title
compares the construction's angle with n₁ sin i = n₂ sin r. The pale lines are wavefronts one wavelength apart: closer
together in the slower material. Set n above bigger than n below and tilt the wave: past the critical angle the
wavelet reaches beyond C, no line can touch it, and no wave gets out.

In [ ]:
def huygens_picture(angle=45.0, n_above=1.0, n_below=1.5):
    H = huygens(angle, n_above, n_below)
    fig, ax = plt.subplots(figsize=(8.4, 5.6))
    fig.subplots_adjust(left=0.04, right=0.98, top=0.86, bottom=0.06)
    AC = H["AC"]
    ax.fill_between([-3, AC + 3], -5.2, 0, color=SKY, alpha=0.10 + 0.12 * (n_below - 1)); ax.axhline(0, color=CHALK, lw=1.5)
    if n_above > 1: ax.fill_between([-3, AC + 3], 0, 4.6, color=SKY, alpha=0.10 + 0.12 * (n_above - 1))
    d_in = np.array([sind(angle), -cosd(angle)]); t_in = np.array([cosd(angle), sind(angle)])
    for k in range(-1, 5):                                              # the arriving wavefronts, one wavelength (1 unit) apart
        p0 = -k * 1.0 * d_in
        s0, s1 = -3.5, 6.0
        pts = np.array([p0 + s * t_in for s in np.linspace(s0, s1, 60)])
        pts = pts[pts[:, 1] >= 0]
        if len(pts): ax.plot(pts[:, 0], pts[:, 1], color=GHOST, lw=1.4)
    ax.plot([0, H["B"][0]], [0, H["B"][1]], color=CHALK, lw=2.6)                                   # the wavefront AB
    ax.plot([H["B"][0], AC], [H["B"][1], 0], color=DIM, lw=1.4, ls="--")                           # B travels to C
    th = np.linspace(math.pi, 2 * math.pi, 120)
    ax.plot(H["AE"] * np.cos(th), H["AE"] * np.sin(th), color=DIM, lw=1.6)                         # the wavelet from A
    for name, pt, off in (("A", (0, 0), (-0.35, 0.15)), ("B", H["B"], (0.1, 0.2)), ("C", (AC, 0), (0.15, 0.15))):
        ax.plot([pt[0]], [pt[1]], "o", color=CHALK, ms=5); ax.text(pt[0] + off[0], pt[1] + off[1], name, color=CHALK, family="monospace")
    if H["r"] is not None:
        E = H["E"]
        ax.plot([AC, E[0]], [0, E[1]], color=GOLD, lw=2.8)                                         # the new wavefront CE
        ax.plot([0, E[0]], [0, E[1]], color=DIM, lw=1.2, ls="--"); ax.plot([E[0]], [E[1]], "o", color=GOLD, ms=6)
        ax.text(E[0] - 0.1, E[1] - 0.45, "E", color=GOLD, family="monospace")
        d_out = np.array([sind(H["r"]), -cosd(H["r"])]); t_out = np.array([cosd(H["r"]), sind(H["r"])])
        lam2 = n_above / n_below                                                                   # the wavelength below, in units of the one above
        for k in range(1, 7):
            pts = np.array([np.array(E) + (k - 1) * lam2 * d_out + s * t_out for s in np.linspace(-6, 6, 80)])
            pts = pts[(pts[:, 1] <= 0) & (pts[:, 1] >= -5.2)]
            if len(pts): ax.plot(pts[:, 0], pts[:, 1], color=GHOST, lw=1.4)
        t = (f"i = {angle:g}°: Huygens' tangent gives r = {H['r']:.2f}° · Snell's law gives {refract(angle, n_above, n_below):.2f}° · "
             f"wavelength below = {lam2:.3f} × the one above")
    else:
        t = (f"i = {angle:g}°: the wavelet (radius {H['AE']:.2f}) has passed C ({AC:.2f} away): no tangent, no wave gets out "
             f"(critical angle {critical(n_above, n_below):.1f}°)")
    ax.set_xlim(-3, AC + 3); ax.set_ylim(-5.2, 4.6); ax.set_aspect("equal"); ax.grid(False); ax.set_xticks([]); ax.set_yticks([])
    fig.suptitle(t, x=0.03, ha="left", color=GOLD, fontsize=11.5)
    footnote(fig, "Huygens' construction · computed"); plt.show()
    return H["r"]

r_ = huygens_picture(45.0)
assert close(r_, refract(45.0, 1.0, 1.5), 1e-9) and round(r_, 2) == 28.13          # the tangent IS Snell's law
assert close(huygens_picture(30.0, 1.5, 1.0), refract(30.0, 1.5, 1.0), 1e-9)        # out of glass: 48.59°
assert huygens_picture(50.0, 1.5, 1.0) is None and round(critical(1.5), 1) == 41.8  # past the critical angle: no tangent
assert agrees("P1.3", [huygens(50.0, 1.0, N_WATER)["r"]]) and agrees("P2.1", [huygens(35.0, N_WATER, 1.0)["r"], critical(N_WATER)])
assert agrees("P3.2", [asind(1.5 * sind(25.0))])

In [ ]:
interact(huygens_picture, angle=FloatSlider(value=45.0, min=5.0, max=80.0, step=1.0, description="angle i (°)"),
         n_above=FloatSlider(value=1.0, min=1.0, max=2.0, step=0.01, description="n above"),
         n_below=FloatSlider(value=1.5, min=1.0, max=2.0, step=0.01, description="n below"));

## Two waves as arrows

Draw each wave as an arrow as long as its amplitude. Set the second at the phase angle to the first, tip to tail.
The arrow that closes the triangle is the amplitude of the sum, and the intensity goes as its square:
I = 4 I₀ cos²(φ/2) for two equal waves. On the right, the two waves and their sum, added point by point: its height is
exactly the closing arrow. In step: four times one wave's intensity. Half a cycle apart: nothing.

In [ ]:
def arrows_picture(phase=60.0):
    amp = math.hypot(1 + cosd(phase), sind(phase)); I = amp ** 2
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), gridspec_kw={"width_ratios": [1, 1.5]})
    fig.subplots_adjust(left=0.05, right=0.98, top=0.84, bottom=0.12, wspace=0.16)
    ax = axes[0]
    ax.annotate("", (1, 0), (0, 0), arrowprops=dict(arrowstyle="->", color=CORAL, lw=2.6))
    ax.annotate("", (1 + cosd(phase), sind(phase)), (1, 0), arrowprops=dict(arrowstyle="->", color=MINT, lw=2.6))
    ax.annotate("", (1 + cosd(phase), sind(phase)), (0, 0), arrowprops=dict(arrowstyle="->", color=GOLD, lw=3.0))
    ax.set_xlim(-0.3, 2.3); ax.set_ylim(-1.2, 1.2); ax.set_aspect("equal"); ax.grid(False); ax.set_xticks([]); ax.set_yticks([])
    ax.set_title("the arrows, tip to tail", loc="left", color=DIM)
    t = np.linspace(0, 2, 400)
    w1, w2 = np.sin(2 * np.pi * t), np.sin(2 * np.pi * t + math.radians(phase))
    ax = axes[1]
    ax.plot(t, w1, color=CORAL, lw=1.6); ax.plot(t, w2, color=MINT, lw=1.6); ax.plot(t, w1 + w2, color=GOLD, lw=2.8)
    ax.axhline(amp, color=DIM, lw=1, ls="--"); ax.set_ylim(-2.2, 2.2); ax.set_xlabel("cycles"); ax.set_title("the waves, and their sum", loc="left", color=DIM)
    assert abs(np.max(np.abs(w1 + w2)) - amp) < 2e-3 or amp < 2e-3                    # the sum's height is the closing arrow
    fig.suptitle(f"phase difference {phase:g}°: amplitude {amp:.3f} × one wave's · intensity {I:.3f} I₀ · "
                 f"4 cos²(φ/2) = {two_wave(phase):.3f}", x=0.03, ha="left", color=GOLD)
    footnote(fig, "computed"); plt.show()
    return I

assert close(arrows_picture(60.0), 3.0, 1e-9) and close(arrows_picture(0.0), 4.0, 1e-9) and arrows_picture(180.0) < 1e-20
assert agrees("P1.6", [arrows_picture(90.0)]) and agrees("P2.5", [two_wave(phase_deg(1, 3))])
assert agrees("P2.3", [phase_deg(30 - 23, 2) % 360]) and agrees("P3.4", [((3 + 1) / (3 - 1)) ** 2])
assert abs(np.mean([two_wave(p) for p in np.arange(0.05, 360, 0.1)]) - 2.0) < 1e-6   # incoherent: every phase in turn: 2 I₀

In [ ]:
interact(arrows_picture, phase=FloatSlider(value=60.0, min=0.0, max=360.0, step=5.0, description="phase (°)"));

## A ripple tank

Two dippers tap the water together. At every point the two ripples add: the picture shows how big the water's swing
is there (bright: the waves arrive in step; dark: they cancel). The dark curves are where the path difference is a
whole number of wavelengths plus a half. Move the dippers apart, or shorten the ripples, and count the calm lines.

In [ ]:
def tank_picture(separation_cm=6.0, wavelength_cm=1.5):
    xs, ys = np.linspace(-14, 14, 420), np.linspace(0.2, 16, 240)
    X, Y = np.meshgrid(xs, ys)
    r1, r2 = np.hypot(X + separation_cm / 2, Y), np.hypot(X - separation_cm / 2, Y)
    swing = np.abs(2 * np.cos(np.pi * (r2 - r1) / wavelength_cm))                  # two equal arrows, added
    fig, ax = plt.subplots(figsize=(9, 5.2))
    fig.subplots_adjust(left=0.07, right=0.98, top=0.86, bottom=0.12)
    ax.imshow(swing, extent=[xs[0], xs[-1], ys[0], ys[-1]], origin="lower", cmap="bone", vmin=0, vmax=2, aspect="equal")
    ax.plot([-separation_cm / 2, separation_cm / 2], [0.2, 0.2], "o", color=GOLD, ms=9); ax.grid(False)
    ax.set_xlabel("cm"); ax.set_ylabel("cm from the dippers")
    calm = sum(1 for n in range(0, 200) if (n + 0.5) * wavelength_cm < separation_cm)
    fig.suptitle(f"dippers {separation_cm:g} cm apart, ripples {wavelength_cm:g} cm long: {calm} calm lines on each side of the middle",
                 x=0.03, ha="left", color=GOLD)
    footnote(fig, "computed: two equal waves added at every point"); plt.show()
    return calm

assert tank_picture(6.0, 1.5) == 4                                                # the course's tank
_r1, _r2 = 12.0, 15.0                                                             # the course's point P: two wavelengths: in step
assert close(abs(2 * math.cos(math.pi * (_r2 - _r1) / 1.5)), 2.0, 1e-9) and abs(2 * math.cos(math.pi * (15.75 - 12.0) / 1.5)) < 1e-9
assert agrees("P1.4", [1250 / 500])

In [ ]:
interact(tank_picture, separation_cm=FloatSlider(value=6.0, min=1.0, max=12.0, step=0.5, description="apart (cm)"),
         wavelength_cm=FloatSlider(value=1.5, min=0.5, max=4.0, step=0.1, description="ripple (cm)"));

## Two slits

Each slit is one source. For every point of the screen the notebook takes the exact distance to each slit, turns it
into an arrow, adds the two arrows and squares the sum: no fringe formula is used. The band at the top is what you
would see; the curve is the intensity. The gold ticks are where the formula x = nλD/d says the bright fringes should
be, and the title compares the fringe width measured on the summed pattern with β = λD/d. Slide the slits apart and
watch the fringes close up; lengthen the wavelength and watch them spread.

In [ ]:
def slits_picture(wavelength_nm=600.0, d_mm=0.3, D_m=1.5):
    lam, d = wavelength_nm * NM, d_mm * MM
    beta = fringe_width(lam, D_m, d)
    xs = np.linspace(-3.6 * beta, 3.6 * beta, 2881)
    I = pattern(xs, slit_sources([d / 2, -d / 2]), lam, D_m)
    pk = peaks(xs, I, 0.5)
    measured = (pk[-1][0] - pk[0][0]) / (len(pk) - 1)
    fig, axes = plt.subplots(2, 1, figsize=(10, 4.6), gridspec_kw={"height_ratios": [1, 2.4]}, sharex=True)
    fig.subplots_adjust(left=0.08, right=0.98, top=0.84, bottom=0.14, hspace=0.08)
    strip(axes[0], xs / MM, I, gamma=1.0)
    axes[1].plot(xs / MM, 4 * I, color=VIOLET)
    for n in range(-3, 4): axes[1].plot([bright(n, lam, D_m, d) / MM], [4.25], "v", color=GOLD, ms=7)
    axes[1].set_xlabel("height on the screen (mm)"); axes[1].set_ylabel("intensity (I₀)"); axes[1].set_ylim(0, 4.6)
    fig.suptitle(f"fringe width of the summed pattern: {measured / MM:.4f} mm · β = λD/d = {beta / MM:.4f} mm",
                 x=0.03, ha="left", color=GOLD)
    footnote(fig); plt.show()
    return measured / MM

b_ = slits_picture()
assert close(b_, 3.0, 5e-3)                                                       # the course's example: 3 mm, within 0.5 %
assert agrees("P1.5", [slits_picture(600.0, 0.5, 2.0)]) and agrees("P1.10", [slits_picture(650.0, 0.5, 3.0)])
assert agrees("P2.4", [bright(4, 500 * NM, 1.5, 0.25 * MM) / MM, dark(1, 500 * NM, 1.5, 0.25 * MM) / MM])
assert agrees("P2.6", [2.6 / N_WATER]) and agrees("P3.3", [11 * MM / 5 * 0.3 * MM / 1.2 / NM]) and agrees("P3.5", [3.9 * 450 / 650])

In [ ]:
interact(slits_picture, wavelength_nm=FloatSlider(value=600.0, min=400.0, max=750.0, step=10.0, description="λ (nm)"),
         d_mm=FloatSlider(value=0.3, min=0.1, max=1.0, step=0.05, description="d (mm)"),
         D_m=FloatSlider(value=1.5, min=0.5, max=5.0, step=0.1, description="D (m)"));

## One slit

Now a single slit, cut into 200 strips. Every strip is a Huygens source; their arrows are added at each point of the
screen and the sum is squared. The band at the top is drawn the way your eye would see it (the faint side bands
lifted). The dashed curve is the textbook shape (sin α/α)²; the gold ticks are the dark lines the course found by
pairing off the slit's halves, x = nλD/a. Narrow the slit and the pattern spreads.

In [ ]:
def slit_picture(wavelength_nm=600.0, a_mm=0.1, D_m=1.5):
    lam, a = wavelength_nm * NM, a_mm * MM
    x1 = slit_dark(1, lam, D_m, a)
    xs = np.linspace(-3.3 * x1, 3.3 * x1, 1981)
    I = pattern(xs, slit_sources([0.0], a, 200), lam, D_m)
    dp = sorted(x_ for x_, _ in dips(xs, I) if x_ > 0)
    fig, axes = plt.subplots(2, 1, figsize=(10, 4.6), gridspec_kw={"height_ratios": [1, 2.4]}, sharex=True)
    fig.subplots_adjust(left=0.08, right=0.98, top=0.84, bottom=0.14, hspace=0.08)
    strip(axes[0], xs / MM, I, gamma=0.4)
    axes[1].plot(xs / MM, I, color=VIOLET); axes[1].plot(xs / MM, slit_shape(xs, lam, D_m, a), color=DIM, lw=1.2, ls="--")
    for n in (-3, -2, -1, 1, 2, 3): axes[1].plot([slit_dark(n, lam, D_m, a) / MM], [0.03], "^", color=GOLD, ms=7)
    axes[1].set_xlabel("height on the screen (mm)"); axes[1].set_ylabel("intensity (middle = 1)")
    fig.suptitle(f"first dark line of the summed pattern: {dp[0] / MM:.4f} mm · λD/a = {x1 / MM:.4f} mm · central maximum "
                 f"{2 * dp[0] / MM:.3f} mm wide", x=0.03, ha="left", color=GOLD)
    footnote(fig); plt.show()
    return dp[0] / MM, 2 * dp[0] / MM, float(np.max(np.abs(I - slit_shape(xs, lam, D_m, a))))

x1_, w_, err_ = slit_picture()
assert close(x1_, 9.0, 5e-3) and close(w_, 18.0, 5e-3) and err_ < 0.005            # 9 mm and 18 mm; the shape is (sin α/α)²
assert agrees("P1.7", [slit_picture(500.0, 0.1, 2.0)[0]]) and agrees("P1.8", [slit_picture(500.0, 0.1, 2.0)[1]])
assert agrees("P2.7", [central_width(RED, 2.0, 0.13 * MM) / MM, slit_dark(2, RED, 2.0, 0.13 * MM) / MM])
assert agrees("P3.6", [2 * 600 * NM * 3.0 / (18 * MM) / MM])
assert close(slit_picture(600.0, 0.05, 1.5)[1], 36.0, 5e-3)                         # half the slit: twice the width

In [ ]:
interact(slit_picture, wavelength_nm=FloatSlider(value=600.0, min=400.0, max=750.0, step=10.0, description="λ (nm)"),
         a_mm=FloatSlider(value=0.1, min=0.02, max=0.5, step=0.01, description="a (mm)"),
         D_m=FloatSlider(value=1.5, min=0.5, max=5.0, step=0.1, description="D (m)"));

## Two real slits

Real slits have width. Give each of the two slits 40 strips and add all 80 arrows: the fine two-slit fringes appear
inside the broad pattern of one slit (the dashed curve). Where one slit's pattern is dark, a fringe goes missing. The
title counts the bright fringes inside the central maximum. With d/a a whole number, that fringe falls exactly on the
dark line.

In [ ]:
def both_picture(wavelength_nm=600.0, d_mm=0.3, a_mm=0.06, D_m=1.5):
    lam, d, a = wavelength_nm * NM, d_mm * MM, a_mm * MM
    edge, beta = slit_dark(1, lam, D_m, a), fringe_width(lam, D_m, d)
    xs = np.linspace(-1.6 * edge, 1.6 * edge, 3201)
    I = pattern(xs, slit_sources([d / 2, -d / 2], a, 40), lam, D_m)
    inside = [p for p in peaks(xs, I, 0.004) if abs(p[0]) < edge - beta / 2]
    fig, axes = plt.subplots(2, 1, figsize=(10, 4.6), gridspec_kw={"height_ratios": [1, 2.4]}, sharex=True)
    fig.subplots_adjust(left=0.08, right=0.98, top=0.84, bottom=0.14, hspace=0.08)
    strip(axes[0], xs / MM, I, gamma=0.5)
    axes[1].plot(xs / MM, I, color=VIOLET); axes[1].plot(xs / MM, slit_shape(xs, lam, D_m, a), color=DIM, lw=1.2, ls="--")
    axes[1].set_xlabel("height on the screen (mm)"); axes[1].set_ylabel("intensity (middle = 1)")
    fig.suptitle(f"d/a = {d / a:.2f}: {len(inside)} bright fringes inside one slit's central maximum ({2 * edge / MM:.1f} mm wide) · "
                 f"β = {beta / MM:.2f} mm", x=0.03, ha="left", color=GOLD)
    footnote(fig); plt.show()
    return len(inside)

assert both_picture() == 9                                                          # the course's example: d/a = 5
assert agrees("P2.8", [both_picture(600.0, 0.4, 0.1, 1.5)])
assert agrees("P3.8", [fringe_width(RED, 2.5, 0.25 * MM) / MM, both_picture(650.0, 0.25, 0.05, 2.5)])

In [ ]:
interact(both_picture, wavelength_nm=FloatSlider(value=600.0, min=400.0, max=750.0, step=10.0, description="λ (nm)"),
         d_mm=FloatSlider(value=0.3, min=0.1, max=1.0, step=0.05, description="d (mm)"),
         a_mm=FloatSlider(value=0.06, min=0.02, max=0.2, step=0.01, description="a (mm)"),
         D_m=FloatSlider(value=1.5, min=0.5, max=5.0, step=0.1, description="D (m)"));

## Tier 2 · Core

The level of the course's "Problems, step by step" chapters.

In [ ]:
practice("P2.1")

In [ ]:
practice("P2.2")

In [ ]:
practice("P2.3")

In [ ]:
practice("P2.4")

In [ ]:
practice("P2.5")

In [ ]:
practice("P2.6")

In [ ]:
practice("P2.7")

In [ ]:
practice("P2.8")

In [ ]:
practice("P2.9")

In [ ]:
practice("P2.10")

### How you'd do this in Python

In [ ]:
print(f"P2.2  v = {speed_in(N_WATER) / 1e8:.2f} × 10⁸ m/s, λ = {lam_in(589, N_WATER):.2f} nm;  "
      f"P2.10  λ = {(43 / 10) * MM * 0.25 * MM / 2.0 / NM:.1f} nm")
assert agrees("P2.1", [refract(35, N_WATER, 1), critical(N_WATER)]) and agrees("P2.2", [speed_in(N_WATER) / 1e8, lam_in(589, N_WATER)])
assert agrees("P2.3", [phase_deg(7, 2) % 360]) and agrees("P2.5", [two_wave(120)]) and agrees("P2.6", [2.6 / N_WATER])
assert agrees("P2.9", [20 / 2 * malus(30)]) and agrees("P2.10", [(43 / 10) * MM * 0.25 * MM / 2.0 / NM])
print("Every tier 2 answer, worked out in Python, matches the practice set.")

## Polaroids

Unpolarised light meets a row of polaroids. The first passes half the intensity, vibrating along its pass axis. Each
later one passes cos² of the angle between its axis and the vibration reaching it (Malus's law), and turns the
vibration to its own axis. Set the second at 90° to the first: nothing. Then give the middle sheet 45° and the last
90°: an eighth comes through.

In [ ]:
def polaroid_picture(angle_2=30.0, angle_3=None):
    axes_ = [0.0, angle_2] + ([] if angle_3 is None else [angle_3])
    I = [1.0, 0.5]                                                                  # before any sheet; after the first
    for k in range(1, len(axes_)): I.append(I[-1] * malus(axes_[k] - axes_[k - 1]))
    fig, ax = plt.subplots(figsize=(8.6, 3.8))
    fig.subplots_adjust(left=0.09, right=0.98, top=0.82, bottom=0.2)
    labels = ["unpolarised"] + [f"after sheet {k + 1}\n(axis {a_:g}°)" for k, a_ in enumerate(axes_)]
    ax.bar(range(len(I)), I, color=VIOLET, width=0.55)
    for k, v in enumerate(I): ax.text(k, v + 0.03, f"{v:.4g}", ha="center", color=CHALK)
    ax.set_xticks(range(len(I))); ax.set_xticklabels(labels); ax.set_ylim(0, 1.15); ax.set_ylabel("intensity (I₀)")
    fig.suptitle(f"{len(axes_)} polaroids: {I[-1]:.4f} of the original intensity gets through", x=0.03, ha="left", color=GOLD)
    footnote(fig, "ideal polaroids · computed"); plt.show()
    return I[-1]

assert close(polaroid_picture(60.0), 0.125, 1e-9) and polaroid_picture(90.0) < 1e-30
assert close(polaroid_picture(45.0, 90.0), 0.125, 1e-9)                            # a third sheet between crossed ones: an eighth
assert agrees("P1.9", [10 * malus(60)]) and agrees("P2.9", [20 * polaroid_picture(30.0)]) and agrees("P3.7", [polaroid_picture(30.0, 90.0)])
assert abs(np.mean([malus(t) for t in np.arange(0.05, 360, 0.1)]) - 0.5) < 1e-6    # unpolarised: every direction: a half

In [ ]:
interact(polaroid_picture, angle_2=FloatSlider(value=30.0, min=0.0, max=90.0, step=5.0, description="sheet 2 (°)"),
         angle_3=FloatSlider(value=90.0, min=0.0, max=180.0, step=5.0, description="sheet 3 (°)"));

## The laser and the two slits

The course's opening question. A red laser pointer shines through two slits 0.50 mm apart, each 0.04 mm wide, onto a
wall 4 m away. The pattern is summed as before (80 arrows at every point). Count eleven stripes in a row, measure
the ten gaps between them with a ruler that reads to the millimetre, and work back: λ = βd/D. Try a green pointer
(532 nm), or your own slide and wall. **If you do it for real: never look into the beam, never aim it at anyone.**

In [ ]:
def laser_picture(wavelength_nm=650.0, d_mm=0.50, a_mm=0.04, D_m=4.0, gaps=10):
    lam, d, a = wavelength_nm * NM, d_mm * MM, a_mm * MM
    beta, edge = fringe_width(lam, D_m, d), slit_dark(1, lam, D_m, a)
    xs = np.linspace(-1.08 * edge, 1.08 * edge, 7001)
    I = pattern(xs, slit_sources([d / 2, -d / 2], a, 40), lam, D_m)
    pk = peaks(xs, I, 0.0002)
    c = min(range(len(pk)), key=lambda k: abs(pk[k][0]))
    half = gaps // 2
    span = pk[c + half][0] - pk[c - half][0]                                       # what the ruler is laid across
    ruler_mm = float(round(span / MM))                                             # a ruler reads to the millimetre
    measured_nm = (ruler_mm * MM / gaps) * d / D_m / NM
    colour = (0.92, 0.25, 0.20) if wavelength_nm > 600 else (0.30, 0.85, 0.35) if wavelength_nm > 495 else (0.35, 0.50, 0.95)
    fig, axes = plt.subplots(2, 1, figsize=(10, 4.6), gridspec_kw={"height_ratios": [1.3, 1]}, sharex=True)
    fig.subplots_adjust(left=0.08, right=0.98, top=0.84, bottom=0.2, hspace=0.1)
    strip(axes[0], xs / MM, I, colour=colour, gamma=0.5)
    axes[1].plot(xs / MM, I, color=VIOLET, lw=1.4); axes[1].plot(xs / MM, slit_shape(xs, lam, D_m, a), color=DIM, lw=1.1, ls="--")
    axes[1].plot([pk[c - half][0] / MM, pk[c + half][0] / MM], [1.08, 1.08], "|-", color=GOLD, ms=12)
    axes[1].set_ylim(0, 1.2); axes[1].set_xlabel("position on the wall (mm)"); axes[1].set_ylabel("intensity")
    fig.suptitle(f"{gaps} gaps span {span / MM:.2f} mm: {ruler_mm:.0f} mm on a ruler → β = {ruler_mm / gaps:.2f} mm → "
                 f"λ = βd/D = {measured_nm:.0f} nm (the pointer: {wavelength_nm:g} nm)", x=0.03, ha="left", color=GOLD)
    footnote(fig); plt.show()
    inside = [p for p in pk if abs(p[0]) < edge]
    return ruler_mm, measured_nm, len(inside)

ruler_, lam_, stripes_ = laser_picture()
assert (ruler_, round(lam_, 6), stripes_) == (52.0, 650.0, 25)                      # 52 mm, 650 nm, 25 stripes in the band
assert close(fringe_width(GREEN, 4.0, 0.5 * MM) / MM, 4.256, 1e-9)                  # a green pointer: 4.26 mm
assert agrees("P2.10", [(43 / 10) * MM * 0.25 * MM / 2.0 / NM])

In [ ]:
interact(laser_picture, wavelength_nm=FloatSlider(value=650.0, min=405.0, max=670.0, step=1.0, description="λ (nm)"),
         d_mm=FloatSlider(value=0.50, min=0.125, max=0.75, step=0.025, description="d (mm)"),
         a_mm=FloatSlider(value=0.04, min=0.02, max=0.16, step=0.02, description="a (mm)"),
         D_m=FloatSlider(value=4.0, min=1.0, max=6.0, step=0.5, description="D (m)"),
         gaps=IntSlider(value=10, min=2, max=10, step=2, description="gaps"));

## Tier 3 · Stretch

Run a formula backwards, combine two ideas, spot the trap.

In [ ]:
practice("P3.1")

In [ ]:
practice("P3.2")

In [ ]:
practice("P3.3")

In [ ]:
practice("P3.4")

In [ ]:
practice("P3.5")

In [ ]:
practice("P3.6")

In [ ]:
practice("P3.7")

In [ ]:
practice("P3.8")

### How you'd do this in Python

In [ ]:
print(f"P3.1  n = {640 / 400:.2f}, v = {C / 1.6 / 1e8:.3f} × 10⁸ m/s;  P3.6  a = {2 * 600 * NM * 3.0 / (18 * MM) / MM:.2f} mm;  "
      f"P3.7  {0.5 * malus(30) * malus(60):.5f}")
assert agrees("P3.1", [640 / 400, C / 1.6 / 1e8]) and agrees("P3.2", [asind(1.5 * sind(25))]) and agrees("P3.3", [11 / 5 * MM * 0.3 * MM / 1.2 / NM])
assert agrees("P3.4", [16 / 4]) and agrees("P3.5", [3.9 * 450 / 650]) and agrees("P3.6", [2 * 600 * NM * 3.0 / (18 * MM) / MM])
assert agrees("P3.7", [0.5 * malus(30) * malus(60)]) and agrees("P3.8", [fringe_width(RED, 2.5, 0.25 * MM) / MM, 9])
print("Every tier 3 answer, worked out in Python, matches the practice set.")

## Sources
Every exercise and set-up here is our own. The refractive indices are the course's rounded values (water 1.33, glass
1.5: OpenStax University Physics volume 3, table 1.1). A red laser pointer is taken as 650 nm and a green one as
532 nm (red consumer lasers emit between 630 and 670 nm: UK Health Security Agency, "Laser radiation: safety advice").
The two-slit slide (slits 0.04 mm wide, 0.50 mm apart, the spacing good to 0.01 mm) is a common teaching slide
(PASCO OS-8453). Every pattern is summed arrow by arrow from the slits' exact distances; the formulas are only
compared with it.

In [ ]:
# When this notebook runs beside the course's source files, this cell checks it still matches them; elsewhere it does
# nothing.
import importlib.util, pathlib, sys
_src = pathlib.Path("../../../episodes/ph20_wave_optics/practice.py")
if _src.exists():
    sys.path.insert(0, str(_src.resolve().parents[2]))
    _spec = importlib.util.spec_from_file_location("ph20_practice", _src)
    _P = importlib.util.module_from_spec(_spec); _spec.loader.exec_module(_P)
    def _flat(a): return [y for x in a for y in _flat(x)] if isinstance(a, (tuple, list)) else [a]
    for _e in _P.EX:
        _q = Q[_e["id"]]
        assert (_q["q"], _q["working"], _q["function"], _q["tier"], _q["parts"], _q["hint"]) == (
            _e["q"], _e["working"], _e["function"], _e["tier"], _e["parts"], _e["hint"]), _e["id"]
        assert len(_q["answer"]) == len(_flat(_e["answer"])) and all(
            abs(a - float(b)) < 1e-9 * max(1, abs(b)) for a, b in zip(_q["answer"], _flat(_e["answer"]))), _e["id"]
    assert len(_P.EX) == len(EX)
    assert (N_WATER, N_GLASS, C, RED, GREEN) == (_P.N_WATER, _P.N_GLASS, _P.C, _P.RED, _P.GREEN)
    _xs = _P.PATTERNS["y_pattern"]["x"][::200]                                     # the notebook's pattern sum is the course's
    _mine = pattern(_xs, slit_sources([_P.YD["d"] / 2, -_P.YD["d"] / 2]), _P.YD["lam"], _P.YD["D"])
    assert max(abs(a - b) for a, b in zip(_mine, _P.PATTERNS["y_pattern"]["I"][::200])) < 1e-6
    assert close(huygens(45.0, 1.0, 1.5)["r"], _P.HUY["refract"]["r"], 1e-12)
    assert (ruler_, round(lam_, 6), stripes_) == (round(_P.LASER["ruler"] / _P.MM, 9), round(_P.LASER["lam_measured"] / _P.NM, 6), len(_P.LASER["inside"]))
    print("matches the course's practice set")